### A Small Variation on an Existing Strategy

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

In [2]:
START_DATE = "2025-01-01"
END_DATE   = "2025-12-31"
DOWNLOAD_START = "2024-01-01"   # 多下一年用于 lag
ONE_WAY_TCOST = 0.0005
TRADING_DAYS = 252

In [3]:
symbols_url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"
symbols_df = pd.read_csv(symbols_url)
symbols = symbols_df["Symbol"].dropna().tolist()
# yfinance 用 '-' 代替 '.'
symbols = [s.replace(".", "-") for s in symbols]
len(symbols)

503

In [31]:
price_data = yf.download(symbols, period="2y")

/tmp/ipykernel_37448/708573861.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  price_data = yf.download(symbols, period="2y")
[*********************100%***********************]  503 of 503 completed

3 Failed downloads:
['UAL']: SSLError('Failed to perform, curl: (35) TLS connect error: error:00000000:invalid library (0):OPENSSL_internal:invalid library (0). See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
['FI', 'WBA']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')


In [32]:
price_data = price_data.dropna(axis=1, how="all")

In [33]:
close_prices = price_data['Close']
open_prices = price_data['Open']
close_prices

Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2024-01-16,128.595337,181.912048,151.960281,135.020004,109.988914,74.331734,344.038574,597.679993,181.661667,64.188751,...,31.073818,92.632767,56.802845,91.123238,108.452835,65.160004,124.859848,120.404442,243.520004,188.591171
2024-01-17,126.221062,180.970917,152.487274,134.979996,110.153282,74.807182,344.408630,596.099976,179.378693,64.021538,...,30.510035,89.302078,56.038143,90.460968,107.211533,63.900002,124.157700,120.001915,239.169998,186.548813
2024-01-18,128.526398,186.865265,153.672989,137.039993,110.240295,75.082947,350.154480,593.700012,183.770538,63.854340,...,30.622793,92.182152,55.627811,90.293060,108.882896,64.470001,125.485092,121.572739,249.589996,186.580139
2024-01-19,129.275131,189.767838,155.056335,139.929993,110.888077,76.908669,354.390839,611.549988,188.413895,63.343430,...,30.867096,92.309509,55.469273,90.432968,109.342270,65.849998,127.254906,121.111313,249.089996,184.503769
2024-01-22,130.585403,192.076050,155.639771,142.009995,110.404671,76.518806,357.614319,603.590027,190.871017,48.016159,...,31.421482,90.193512,54.956364,90.311714,110.818146,66.690002,126.062187,120.355347,254.570007,187.265701
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-06,147.600006,262.359985,223.929993,138.020004,127.779999,93.919998,275.929993,335.989990,292.940002,60.490002,...,23.990000,120.529999,74.430000,121.050003,141.330002,70.000000,151.410004,94.750000,257.609985,129.350006
2026-01-07,147.809998,260.329987,233.419998,137.039993,127.040001,95.160004,273.980011,338.100006,292.890015,58.520000,...,23.180000,116.370003,73.220001,118.489998,139.429993,71.470001,150.490005,92.669998,246.960007,125.669998
2026-01-08,145.759995,259.040009,224.130005,138.660004,126.180000,96.379997,281.820007,339.040009,299.160004,61.189999,...,23.990000,118.269997,73.379997,122.910004,140.889999,70.160004,153.479996,94.080002,254.639999,126.849998


In [34]:
# =========================
# 5. 日收益率
# =========================
daily_ret = close_prices.pct_change(fill_method=None)

daily_ret_test = daily_ret.iloc[1:, :]
daily_ret_test.loc[:, daily_ret_test.isna().any(axis=0)] #这些是出现的晚或者中途退市的股票

Ticker,GEV,IPG,K,SOLV
Date,,,,
2024-01-17,NaN,-0.011003,0.007203,NaN
2024-01-18,NaN,0.009854,0.000550,NaN
2024-01-19,NaN,0.002518,-0.011547,NaN
2024-01-22,NaN,0.008163,-0.005933,NaN
2024-01-23,NaN,0.005294,0.015669,NaN
...,...,...,...,...
2026-01-06,0.008034,NaN,NaN,0.029208
2026-01-07,-0.034983,NaN,NaN,0.012987
2026-01-08,-0.051214,NaN,NaN,0.018162


In [35]:
# =========================
# 6. 市场等权收益（cross-sectional mean）
# =========================
market_daily_ret = daily_ret.mean(axis=1, skipna=True) #只要这天有一只股票存在，market_ret就不是nan
market_daily_ret

Date
2024-01-16         NaN
2024-01-17   -0.007565
2024-01-18    0.005304
2024-01-19    0.009062
2024-01-22    0.004805
                ...   
2026-01-06    0.011474
2026-01-07   -0.011562
2026-01-08    0.010150
2026-01-09    0.005029
2026-01-12    0.000615
Length: 500, dtype: float64

In [36]:
# =========================
# 7. 构造均值回归权重（市场中性）
# =========================
# 每天可交易股票数量
valid_counts = close_prices.notna().sum(axis=1)
valid_counts

Date
2024-01-16    498
2024-01-17    498
2024-01-18    498
2024-01-19    498
2024-01-22    498
             ... 
2026-01-06    498
2026-01-07    498
2026-01-08    498
2026-01-09    498
2026-01-12    498
Length: 500, dtype: int64

In [37]:
# 均值回归：做空强的，做多弱的
weights = -(
    daily_ret.sub(market_daily_ret, axis=0)
).div(valid_counts, axis=0)

# 不可交易日权重清零（停牌 / 首日）
mask = close_prices.isna() | close_prices.shift(1).isna()
weights[mask] = 0.0

# 这些位置的weights本来是nan，被强制置0了，就是之前打印的出现晚或者退市了的四只股票，所以在True
# 的时候，这些股票在那天是不可交易的
mask.loc[:, mask.sum(axis=0)!=1] 

Ticker,GEV,IPG,K,SOLV
Date,,,,
2024-01-16,True,True,True,True
2024-01-17,True,False,False,True
2024-01-18,True,False,False,True
2024-01-19,True,False,False,True
2024-01-22,True,False,False,True
...,...,...,...,...
2026-01-06,False,True,True,False
2026-01-07,False,True,True,False
2026-01-08,False,True,True,False


In [42]:
# =========================
# 8. 每日 PnL（无交易成本）
# =========================
# 用 t-1 的权重吃 t 的收益
daily_pnl = (weights.shift(1) * (close_prices - open_prices) / open_prices).sum(axis=1)
daily_pnl

Date
2024-01-16    0.000000
2024-01-17    0.000000
2024-01-18   -0.000022
2024-01-19   -0.000012
2024-01-22    0.000034
                ...   
2026-01-06   -0.000018
2026-01-07   -0.000038
2026-01-08    0.000106
2026-01-09   -0.000009
2026-01-12   -0.000048
Length: 500, dtype: float64

In [43]:
# =========================
# 9. 回测区间
# =========================
daily_pnl_2025 = daily_pnl.loc[START_DATE:END_DATE]
weights_2025   = weights.loc[START_DATE:END_DATE]

In [44]:
# =========================
# 10. Sharpe（无手续费）
# =========================
sharpe_no_tc = (
    np.sqrt(TRADING_DAYS)
    * daily_pnl_2025.mean()
    / daily_pnl_2025.std()
)

print("Sharpe (no transaction cost):", sharpe_no_tc)

Sharpe (no transaction cost): 1.3059263036159607


In [45]:
# =========================
# 11. 加入交易成本
# =========================
# 日换手率
turnover = (weights_2025 - weights_2025.shift(1)).abs().sum(axis=1)

daily_pnl_tc = daily_pnl_2025 - turnover * ONE_WAY_TCOST

# =========================
# 12. Sharpe（含手续费）
# =========================
sharpe_with_tc = (
    np.sqrt(TRADING_DAYS)
    * daily_pnl_tc.mean()
    / daily_pnl_tc.std()
)

print("Sharpe (with transaction cost):", sharpe_with_tc)

Sharpe (with transaction cost): -0.4855056063221827
